In [1]:
"""
CIF文件处理脚本
功能：
1. 去除飘散的氢原子和氢气分子
2. 对所有原子的z轴位置添加随机扰动
"""

import numpy as np
from ase.io import read, write
from ase.neighborlist import neighbor_list
from scipy.spatial.distance import cdist
import argparse
import os


def find_isolated_hydrogens(atoms, h_bond_cutoff=1.3, h2_bond_cutoff=0.9):
    """
    识别飘散的氢原子和氢气分子
    
    参数:
        atoms: ASE Atoms对象
        h_bond_cutoff: H与非H原子成键的最大距离（埃）
        h2_bond_cutoff: H2分子中H-H键的最大距离（埃）
    
    返回:
        需要删除的原子索引列表
    """
    positions = atoms.get_positions()
    symbols = atoms.get_chemical_symbols()
    
    # 获取氢原子和非氢原子的索引
    h_indices = [i for i, s in enumerate(symbols) if s == 'H']
    non_h_indices = [i for i, s in enumerate(symbols) if s != 'H']
    
    if not h_indices or not non_h_indices:
        return []
    
    h_positions = positions[h_indices]
    non_h_positions = positions[non_h_indices]
    
    # 计算考虑周期性边界条件的距离
    cell = atoms.get_cell()
    pbc = atoms.get_pbc()
    
    isolated_h = []
    
    for i, h_idx in enumerate(h_indices):
        h_pos = positions[h_idx]
        
        # 计算与所有非氢原子的最小距离（考虑周期性）
        min_dist_to_non_h = float('inf')
        for j in non_h_indices:
            dist = atoms.get_distance(h_idx, j, mic=True)
            min_dist_to_non_h = min(min_dist_to_non_h, dist)
        
        # 如果与非氢原子距离太远，标记为飘散
        if min_dist_to_non_h > h_bond_cutoff:
            isolated_h.append(h_idx)
    
    # 识别H2分子（两个相邻的飘散氢原子）
    to_remove = set()
    used = set()
    
    for i, h1_idx in enumerate(isolated_h):
        if h1_idx in used:
            continue
        for h2_idx in isolated_h:
            if h2_idx <= h1_idx or h2_idx in used:
                continue
            dist = atoms.get_distance(h1_idx, h2_idx, mic=True)
            if dist < h2_bond_cutoff:
                # 这是一个H2分子
                to_remove.add(h1_idx)
                to_remove.add(h2_idx)
                used.add(h1_idx)
                used.add(h2_idx)
                break
        else:
            # 单独的飘散氢原子
            to_remove.add(h1_idx)
    
    return sorted(list(to_remove))


def add_z_perturbation(atoms, amplitude=0.05, seed=None):
    """
    对所有原子的z坐标添加随机扰动
    
    参数:
        atoms: ASE Atoms对象
        amplitude: 扰动幅度（埃）
        seed: 随机种子（可选）
    
    返回:
        修改后的Atoms对象
    """
    if seed is not None:
        np.random.seed(seed)
    
    positions = atoms.get_positions()
    n_atoms = len(atoms)
    
    # 生成[-amplitude, amplitude]范围内的随机扰动
    z_perturbation = np.random.uniform(-amplitude, amplitude, n_atoms)
    
    # 只修改z坐标
    positions[:, 2] += z_perturbation
    
    atoms.set_positions(positions)
    return atoms


def process_cif(input_file, output_file=None, z_amplitude=0.05, 
                h_bond_cutoff=1.3, h2_bond_cutoff=0.9, seed=None):
    """
    处理CIF文件的主函数
    
    参数:
        input_file: 输入CIF文件路径
        output_file: 输出CIF文件路径（默认为input_processed.cif）
        z_amplitude: z轴扰动幅度（埃）
        h_bond_cutoff: H与非H原子成键判断距离
        h2_bond_cutoff: H2分子键长判断距离
        seed: 随机种子
    """
    # 读取CIF文件
    print(f"读取文件: {input_file}")
    atoms = read(input_file)
    
    print(f"原始原子数: {len(atoms)}")
    print(f"原始化学式: {atoms.get_chemical_formula()}")
    
    # 步骤1: 识别并移除飘散的氢原子
    isolated_h = find_isolated_hydrogens(atoms, h_bond_cutoff, h2_bond_cutoff)
    
    if isolated_h:
        print(f"\n发现 {len(isolated_h)} 个飘散的氢原子，索引: {isolated_h}")
        # 创建保留原子的掩码
        mask = [i not in isolated_h for i in range(len(atoms))]
        atoms = atoms[mask]
        print(f"移除后原子数: {len(atoms)}")
        print(f"移除后化学式: {atoms.get_chemical_formula()}")
    else:
        print("\n未发现飘散的氢原子")
    
    # 步骤2: 添加z轴随机扰动
    print(f"\n对所有原子添加z轴扰动 (幅度: ±{z_amplitude} Å)")
    atoms = add_z_perturbation(atoms, amplitude=z_amplitude, seed=seed)
    
    # 保存结果
    if output_file is None:
        base, ext = os.path.splitext(input_file)
        output_file = f"{base}_processed{ext}"
    
    write(output_file, atoms)
    print(f"\n结果已保存至: {output_file}")
    
    return atoms


def main():
    parser = argparse.ArgumentParser(
        description='处理CIF文件：移除飘散氢原子，添加z轴扰动'
    )
    parser.add_argument('input', help='输入CIF文件路径')
    parser.add_argument('-o', '--output', help='输出CIF文件路径')
    parser.add_argument('-z', '--z-amplitude', type=float, default=0.05,
                        help='z轴扰动幅度（埃），默认0.05')
    parser.add_argument('--h-cutoff', type=float, default=1.3,
                        help='H与非H原子成键判断距离（埃），默认1.3')
    parser.add_argument('--h2-cutoff', type=float, default=0.9,
                        help='H2分子键长判断距离（埃），默认0.9')
    parser.add_argument('--seed', type=int, help='随机种子')
    
    args = parser.parse_args()
    
    process_cif(
        input_file=args.input,
        output_file=args.output,
        z_amplitude=args.z_amplitude,
        h_bond_cutoff=args.h_cutoff,
        h2_bond_cutoff=args.h2_cutoff,
        seed=args.seed
    )


In [ ]:
import glob
allcif=glob.glob('dftb_opted_cifs/*.cif')
for cifnam in allcif:
    #cifnam='dftb_opted_cifs\\ald_39_ami_27_dftb_opted.cif'
    newcifnam=cifnam.split('\\')[-1]
    atoms = process_cif(
        input_file=cifnam,
        output_file=newcifnam,
        z_amplitude=0.05,
        seed=42
    )

['dftb_opted_cifs\\ald_10_ami_10_dftb_opted.cif',
 'dftb_opted_cifs\\ald_10_ami_11_dftb_opted.cif',
 'dftb_opted_cifs\\ald_10_ami_12_dftb_opted.cif',
 'dftb_opted_cifs\\ald_10_ami_13_dftb_opted.cif',
 'dftb_opted_cifs\\ald_10_ami_14_dftb_opted.cif',
 'dftb_opted_cifs\\ald_10_ami_15_dftb_opted.cif',
 'dftb_opted_cifs\\ald_10_ami_16_dftb_opted.cif',
 'dftb_opted_cifs\\ald_10_ami_17_dftb_opted.cif',
 'dftb_opted_cifs\\ald_10_ami_18_dftb_opted.cif',
 'dftb_opted_cifs\\ald_10_ami_19_dftb_opted.cif',
 'dftb_opted_cifs\\ald_10_ami_1_dftb_opted.cif',
 'dftb_opted_cifs\\ald_10_ami_20_dftb_opted.cif',
 'dftb_opted_cifs\\ald_10_ami_21_dftb_opted.cif',
 'dftb_opted_cifs\\ald_10_ami_22_dftb_opted.cif',
 'dftb_opted_cifs\\ald_10_ami_23_dftb_opted.cif',
 'dftb_opted_cifs\\ald_10_ami_24_dftb_opted.cif',
 'dftb_opted_cifs\\ald_10_ami_25_dftb_opted.cif',
 'dftb_opted_cifs\\ald_10_ami_26_dftb_opted.cif',
 'dftb_opted_cifs\\ald_10_ami_27_dftb_opted.cif',
 'dftb_opted_cifs\\ald_10_ami_28_dftb_opted.cif',
 